In [2]:
import pyspark

In [3]:
pyspark.__file__

'/home/gofhilman/data-engineering-zoomcamp/batch-processing-homework/.venv/lib/python3.11/site-packages/pyspark/__init__.py'

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("batch-processing") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 13:47:11 WARN Utils: Your hostname, gofhilmanSL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/05 13:47:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 13:47:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [42]:
df = spark.read.parquet("data/yellow_tripdata_2025-11.parquet")

In [43]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [44]:
df = df.repartition(4)

In [10]:
df.write.parquet("output/yellow_2025_11")

In [11]:
import os

output_path = "output/yellow_2025_11"
parquet_files = [f for f in os.listdir(output_path) if f.endswith(".parquet")]
sizes_mb = [os.path.getsize(os.path.join(output_path, f)) / (1024 * 1024) for f in parquet_files]
average_size_mb = sum(sizes_mb) / len(sizes_mb)

print("Average Parquet file size (MB):", average_size_mb)


Average Parquet file size (MB): 24.40898299217224


In [45]:
df.createOrReplaceTempView("trips_data")

In [46]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [ ]:
specific_count = spark \
    .sql("""
        SELECT COUNT(1)
        FROM trips_data
        WHERE DATE(tpep_pickup_datetime) = DATE '2025-11-15'                       
    """) \
    .show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [ ]:
longest_trip = spark \
    .sql("""
        SELECT (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS trip_hours
        FROM trips_data
        ORDER BY trip_hours DESC
        LIMIT 1;
    """) \
    .show()

+-----------------+
|       trip_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [38]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('data/taxi_zone_lookup.csv', inferSchema=True)

In [39]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [40]:
df_zones.schema

StructType([StructField('LocationID', IntegerType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [41]:
df_zones.write.parquet("output/zones", mode="overwrite")

In [49]:
df_join = df \
    .join(df_zones.select("LocationID", "Zone"), df.PULocationID == df_zones.LocationID) \
    .withColumnRenamed("Zone", "PUZone") \
    .drop("LocationId")

In [59]:
df_join.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|              PUZone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              

In [60]:
df_join.createOrReplaceTempView("joined_trips_data")

In [64]:
import pandas

In [65]:
least_PUZone = spark \
    .sql("""
        SELECT PUZone, COUNT(1) AS trips_count
        FROM joined_trips_data
        GROUP BY 1
        ORDER BY 2
        LIMIT 5
    """) \
    .toPandas()

In [66]:
least_PUZone

,PUZone,trips_count
0,Governor's Island/Ellis Island/Liberty Island,1
1,Eltingville/Annadale/Prince's Bay,1
2,Arden Heights,1
3,Port Richmond,3
4,Rossville/Woodrow,4


In [67]:
spark.stop()